In [1]:
import pandas as pd
import re
import json
import time
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:10])  # should print something like sk-xxxxx

client = OpenAI(api_key=api_key)

sk-proj-9d


In [3]:
df = pd.read_csv("information_and_organization_journal_articles.csv")

df.head()

,URL,Journal_Title,Article_Title,Volume_Issue,Month_Year,Abstract,Keywords,Author_name,Author_email,Author_Address
0,https://www.sciencedirect.com/science/article/...,Information and Organization,Expert-AI pairings: Expert interventions in AI...,"Volume 34, Issue 4",December 2024,•\nExpert-AI pairings are practices of combini...,[],Ignacio Fernandez Cruz,ignacio@northwestern.edu,"School of Communication, Northwestern Universi..."
1,https://www.sciencedirect.com/science/article/...,Information and Organization,Beyond connectivity: Artificial intelligence a...,"Volume 34, Issue 4",December 2024,•\nBy creating a digital space where users int...,"['Artificial intelligence', 'Machine learning'...",José F.P. dos Santos,NaN,Affilliated Professor of Practice in Global Ma...
2,https://www.sciencedirect.com/science/article/...,Information and Organization,Beyond connectivity: Artificial intelligence a...,"Volume 34, Issue 4",December 2024,•\nBy creating a digital space where users int...,"['Artificial intelligence', 'Machine learning'...",Peter J. Williamson,p.williamson@jbs.cam.ac.uk,Affilliated Professor of Practice in Global Ma...
3,https://www.sciencedirect.com/science/article/...,Information and Organization,The same but different: Understanding variatio...,"Volume 34, Issue 4",December 2024,"•\nThe initiation, selection, design, implemen...",[],Yennef Vereycken,yennef.vereycken@kuleuven.be,"KU Leuven, Belgium"
4,https://www.sciencedirect.com/science/article/...,Information and Organization,The same but different: Understanding variatio...,"Volume 34, Issue 4",December 2024,"•\nThe initiation, selection, design, implemen...",[],Anne Guisset,anne.guisset@kuleuven.be,"KU Leuven, Belgium"


In [4]:
df.columns.tolist()

['URL',
 'Journal_Title',
 'Article_Title',
 'Volume_Issue',
 'Month_Year',
 'Abstract',
 'Keywords',
 'Author_name',
 'Author_email',
 'Author_Address']

In [5]:
df_clean = pd.DataFrame()

In [6]:
# URL
df_clean["URL"] = df["URL"]

# Journal
df_clean["Journal_Title"] = df["Journal_Title"]

# Title → standardized
df_clean["Standardized_Title"] = (
    df["Article_Title"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Date
df_clean["month_year"] = df["Month_Year"]

# Text fields
df_clean["Abstract"] = df["Abstract"]
df_clean["Keywords"] = df["Keywords"]

# Author
df_clean["Author_Name"] = df["Author_name"]

In [7]:
df_clean.head()

,URL,Journal_Title,Standardized_Title,month_year,Abstract,Keywords,Author_Name
0,https://www.sciencedirect.com/science/article/...,Information and Organization,expert-ai pairings: expert interventions in ai...,December 2024,•\nExpert-AI pairings are practices of combini...,[],Ignacio Fernandez Cruz
1,https://www.sciencedirect.com/science/article/...,Information and Organization,beyond connectivity: artificial intelligence a...,December 2024,•\nBy creating a digital space where users int...,"['Artificial intelligence', 'Machine learning'...",José F.P. dos Santos
2,https://www.sciencedirect.com/science/article/...,Information and Organization,beyond connectivity: artificial intelligence a...,December 2024,•\nBy creating a digital space where users int...,"['Artificial intelligence', 'Machine learning'...",Peter J. Williamson
3,https://www.sciencedirect.com/science/article/...,Information and Organization,the same but different: understanding variatio...,December 2024,"•\nThe initiation, selection, design, implemen...",[],Yennef Vereycken
4,https://www.sciencedirect.com/science/article/...,Information and Organization,the same but different: understanding variatio...,December 2024,"•\nThe initiation, selection, design, implemen...",[],Anne Guisset


In [8]:
def clean_author(name):
    if pd.isna(name):
        return None
    
    name = name.strip()
    
    # remove dots
    name = name.replace(".", "")
    
    # remove brackets
    name = re.sub(r"[()]", "", name)
    
    # normalize spaces
    name = re.sub(r"\s+", " ", name)
    
    parts = name.split()
    
    # fix initials like PK → P K
    if len(parts) > 1:
        first = parts[0]
        if first.isalpha() and first.isupper() and len(first) <= 3:
            parts[0] = " ".join(list(first))
    
    return " ".join(parts)

In [9]:
df_clean["Standardized_Author"] = df_clean["Author_Name"].apply(clean_author)
df_clean["Standardized_Author"] = df_clean["Standardized_Author"].str.title()

In [10]:
df_clean[["Author_Name", "Standardized_Author"]].head(10)

,Author_Name,Standardized_Author
0,Ignacio Fernandez Cruz,Ignacio Fernandez Cruz
1,José F.P. dos Santos,José Fp Dos Santos
2,Peter J. Williamson,Peter J Williamson
3,Yennef Vereycken,Yennef Vereycken
4,Anne Guisset,Anne Guisset
5,Monique Ramioul,Monique Ramioul
6,Christine Abdalla Mikhaeil,Christine Abdalla Mikhaeil
7,Daniel Robey,Daniel Robey
8,Joao Cunha,Joao Cunha
9,Rick Sullivan,Rick Sullivan


In [11]:
cache_file = "affiliation_cache.json"

try:
    with open(cache_file, "r") as f:
        clean_map = json.load(f)
except:
    clean_map = {}

In [12]:
def clean_address(addr):
    if addr in clean_map:
        return clean_map[addr]
    
    prompt = f"""
    Extract university, state, and country from:

    "{addr}"

    Return ONLY valid JSON like:
    {{
        "university": "",
        "state": "",
        "country": ""
    }}
    """
    
    for attempt in range(2):
        try:
            res = client.chat.completions.create(
                model="gpt-4o-mini",
                temperature=0,
                messages=[{"role": "user", "content": prompt}]
            )
            
            content = res.choices[0].message.content.strip()

            # remove markdown
            if "```" in content:
                content = content.split("```")[1]

            # try direct parse
            try:
                output = json.loads(content)
            except:
                start = content.find("{")
                end = content.rfind("}") + 1
                output = json.loads(content[start:end])

            return output
        
        except Exception:
            time.sleep(1)
    
    return {"university": None, "state": None, "country": None}

In [13]:
unique_addresses = df["Author_Address"].dropna().unique()
len(unique_addresses)


610

In [14]:
for i, addr in enumerate(unique_addresses):
    
    if addr not in clean_map:
        result = clean_address(addr)
        
        if result["university"] is not None:
            clean_map[addr] = result
            time.sleep(0.5)  # avoid rate limit
        else:
            print(f"Failed: {addr}")
    
    # save progress every 20
    if i % 20 == 0:
        with open(cache_file, "w") as f:
            json.dump(clean_map, f)
        print(f"Saved at {i}/{len(unique_addresses)}")

# final save
with open(cache_file, "w") as f:
    json.dump(clean_map, f)

print("Done!")

Saved at 0/610
Saved at 20/610
Failed: Hertie School, 180 Friedrichstraße, 10117 Berlin, Germany and Stanford University, 59 Nathan Abbott Way, 94305 Stanford CA, USA
Saved at 40/610
Saved at 60/610
Saved at 80/610
Saved at 100/610
Saved at 120/610
Saved at 140/610
Saved at 160/610
Failed: University of Alberta Business School, University of Alberta, Edmonton AB Canada; Haskayne School of Business, University of Calgary, Calgary AB, Canada
Saved at 180/610
Saved at 200/610
Saved at 220/610
Saved at 240/610
Saved at 260/610
Saved at 280/610
Saved at 300/610
Saved at 320/610
Saved at 340/610
Saved at 360/610
Saved at 380/610
Saved at 400/610
Saved at 420/610
Saved at 440/610
Saved at 460/610
Saved at 480/610
Saved at 500/610
Saved at 520/610
Saved at 540/610
Saved at 560/610
Saved at 580/610
Saved at 600/610
Done!


In [15]:
clean_map["Hertie School, 180 Friedrichstraße, 10117 Berlin, Germany and Stanford University, 59 Nathan Abbott Way, 94305 Stanford CA, USA"] = {
    "university": "Stanford University",
    "state": "California",
    "country": "United States"
}

clean_map["University of Alberta Business School, University of Alberta, Edmonton AB Canada; Haskayne School of Business, University of Calgary, Calgary AB, Canada"] = {
    "university": "University of Calgary",
    "state": "Alberta",
    "country": "Canada"
}

In [16]:
df_clean["Author_University"] = df["Author_Address"].map(
    lambda x: clean_map.get(x, {}).get("university")
)

df_clean["Standardized_University"] = df_clean["Author_University"]

df_clean["Author_State"] = df["Author_Address"].map(
    lambda x: clean_map.get(x, {}).get("state")
)

df_clean["Author_Country"] = df["Author_Address"].map(
    lambda x: clean_map.get(x, {}).get("country")
)

In [18]:
pd.concat([
    df["Author_Address"],
    df_clean[["Author_University", "Author_State", "Author_Country"]]
], axis=1).head(10)

,Author_Address,Author_University,Author_State,Author_Country
0,"School of Communication, Northwestern Universi...",Northwestern University,IL,United States
1,Affilliated Professor of Practice in Global Ma...,INSEAD,,France
2,Affilliated Professor of Practice in Global Ma...,INSEAD,,France
3,"KU Leuven, Belgium",KU Leuven,,Belgium
4,"KU Leuven, Belgium",KU Leuven,,Belgium
5,"KU Leuven, Belgium",KU Leuven,,Belgium
6,"IESEG School of Management, Univ. Lille, CNRS,...",IESEG School of Management,Hauts-de-France,France
7,"Georgia State University, 55 Park Place, Post ...",Georgia State University,GA,USA
8,"IESEG School of Management, 3, Rue De La Digue...",IESEG School of Management,Hauts-de-France,France
9,"The University of Sydney, Australia",The University of Sydney,,Australia


In [19]:
required_cols = [
    "URL",
    "Journal_Title",
    "Standardized_Title",
    "month_year",
    "Abstract",
    "Keywords",
    "Author_Name",
    "Standardized_Author",
    "Author_University",
    "Standardized_University",
    "Author_State",
    "Author_Country"
]

df_clean = df_clean[required_cols]


In [20]:
df_clean["Author_Country"] = df_clean["Author_Country"].replace({
    "USA": "United States",
    "U.S.A": "United States"
})

In [21]:
df_clean.to_csv("information_and_organization_cleaned.csv", index=False)

In [22]:
df_clean["Standardized_Author"].value_counts().head(20)

Standardized_Author
Neil Pollock            9
Robin Williams          8
Michael Barrett         7
Kalle Lyytinen          7
Sundeep Sahay           7
Panos Constantinides    6
Paul M Leonardi         6
Daniel Robey            6
Susan V Scott           5
Emmanuelle Vaast        5
Jonny Holmström         5
John Mingers            5
Lars Mathiassen         4
Rudy Hirschheim         4
Elizabeth Davidson      4
Robin Teigland          4
Ulrike Schultze         4
Eivor Oborn             4
Wanda J Orlikowski      4
Adrian Yeow             3
Name: count, dtype: int64

In [23]:
df_clean["Standardized_Author"].value_counts()

Standardized_Author
Neil Pollock           9
Robin Williams         8
Michael Barrett        7
Kalle Lyytinen         7
Sundeep Sahay          7
                      ..
Alain Pinsonneault     1
Dorit Nevo             1
Saggi Nevo             1
Olivera Marjanovic     1
Richard J Boland Jr    1
Name: count, Length: 577, dtype: int64

In [24]:
df_clean[df_clean["Standardized_Author"].str.contains("Leonardi", na=False)]

,URL,Journal_Title,Standardized_Title,month_year,Abstract,Keywords,Author_Name,Standardized_Author,Author_University,Standardized_University,Author_State,Author_Country
211,https://www.sciencedirect.com/science/article/...,Information and Organization,on the making of crystal balls: five lessons a...,March 2021,•\nSimulations Models are central in the COVID...,[],Paul M. Leonardi,Paul M Leonardi,"University of California, Santa Barbara","University of California, Santa Barbara",California,United States
321,https://www.sciencedirect.com/science/article/...,Information and Organization,a critical approach to human helping in inform...,September 2018,•\nWe explore heteromation (the reliance on us...,[],Paul M. Leonardi,Paul M Leonardi,"University of California, Santa Barbara","University of California, Santa Barbara",CA,United States
373,https://www.sciencedirect.com/science/article/...,Information and Organization,the social media revolution: sharing and learn...,March 2017,•\nSocial media are ushering in an era of leak...,[],Paul M. Leonardi,Paul M Leonardi,UC Santa Barbara,UC Santa Barbara,California,United States
479,https://www.sciencedirect.com/science/article/...,Information and Organization,theoretical foundations for the study of socio...,April 2013,This paper compares two alternative theoretica...,[],Paul M. Leonardi,Paul M Leonardi,Northwestern University,Northwestern University,IL,United States
524,https://www.sciencedirect.com/science/article/...,Information and Organization,knowledge management technology as a stage for...,January 2012,This article explores why it is often difficul...,[],Paul M. Leonardi,Paul M Leonardi,Northwestern University,Northwestern University,IL,United States
619,https://www.sciencedirect.com/science/article/...,Information and Organization,materiality and change: challenges to building...,2008,Researchers have had difficulty accommodating ...,[],Paul M. Leonardi,Paul M Leonardi,Northwestern University,Northwestern University,IL,United States


In [25]:
df_clean["Standardized_Author"].str.lower().value_counts().head(20)

Standardized_Author
neil pollock            9
robin williams          8
michael barrett         7
kalle lyytinen          7
sundeep sahay           7
panos constantinides    6
paul m leonardi         6
daniel robey            6
susan v scott           5
emmanuelle vaast        5
jonny holmström         5
john mingers            5
lars mathiassen         4
rudy hirschheim         4
elizabeth davidson      4
robin teigland          4
ulrike schultze         4
eivor oborn             4
wanda j orlikowski      4
adrian yeow             3
Name: count, dtype: int64

In [26]:
pd.crosstab(
    df_clean["Standardized_Author"],
    df_clean["Author_University"]
).head()

Author_University,,Aalborg University,Aarhus University,Athabasca University,Audencia Business School,"Audencia, Nantes School of Management",Australian National University,Barna Business School,"Baruch College, CUNY",Bentley College,...,Wageningen University & Research,Western University,Western Washington University,William & Mary,Worcester Polytechnic Institute,Wright State University,Yale School of Management,York University,Zagazig University,Zurich University of Applied Sciences
Standardized_Author,,,,,,,,,,,,,,,,,,,,,
A Arora,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A James Baroody,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Aaron Baird,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Abhijit Gopal,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Adrian Yeow,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
# 1. Country normalization
df_clean["Author_Country"] = df_clean["Author_Country"].replace({
    "USA": "United States",
    "U.S.A": "United States",
    "US": "United States"
})

# 2. University trim
df_clean["Standardized_University"] = (
    df_clean["Standardized_University"]
    .str.strip()
    .str.replace(r"\bThe\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

# 3. Remove duplicates
df_clean = df_clean.drop_duplicates(subset=["URL", "Author_Name"])

In [28]:
df_clean.to_csv("information_and_organization_cleaned.csv", index=False)

In [30]:
df_main = pd.read_csv("../../All_Journals_CLEANED.csv")

df_new = pd.read_csv("information_and_organization_cleaned.csv")

df_final = pd.concat([df_main, df_new], ignore_index=True)

df_final = df_final.drop_duplicates()

df_final.to_csv("All_Journals_CLEANED.csv", index=False)
df_final.to_excel("All_Journals_CLEANED.xlsx", index=False)